<a href="https://colab.research.google.com/github/Haleemahafeez/SWYNEX-Intelligent-Feature-/blob/main/SWYNEX_Intelligent_Feature.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


import sys
import subprocess
import getpass
import time

# Install the official Gemini SDK
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "google-genai"
])

from google import genai

print("AI Study Assistant - SWYNEX Task 3")
print("=" * 45)

# Ask for API key securely; it will not be printed
API_KEY = getpass.getpass("Paste your Gemini API key (hidden): ").strip()

if not API_KEY:
    raise ValueError("API key is empty. Run the cell again and enter your key.")

client = genai.Client(api_key=API_KEY)

# Try commonly available Gemini models
MODEL_CANDIDATES = [
    "gemini-2.5-flash",
    "gemini-2.0-flash",
    "gemini-3-flash-preview",
]

MODEL = None

print("\nChecking Gemini model access...")

for candidate in MODEL_CANDIDATES:
    try:
        response = client.models.generate_content(
            model=candidate,
            contents="Reply with exactly: Connection successful!"
        )
        if response.text:
            MODEL = candidate
            print("Connected successfully.")
            print("Selected model:", MODEL)
            break
    except Exception:
        continue

if MODEL is None:
    raise RuntimeError(
        "Could not connect to the listed models. Check your API key, "
        "model availability, internet connection and API quota, then retry."
    )

SYSTEM_PROMPT = """
You are a friendly AI Study Assistant for students.
Explain concepts in simple English.
Give step-by-step explanations when useful.
For difficult topics, provide a short example.
If a question is unclear, ask the student to clarify it.
Do not claim that an answer is certainly correct if you are unsure.
Encourage students to learn instead of just memorizing.
"""

def generate_answer(question, retries=2):
    """Validate input, generate an answer and handle temporary errors."""

    if not isinstance(question, str) or not question.strip():
        return "Please enter a study question first."

    prompt = SYSTEM_PROMPT + "\nStudent question: " + question.strip()

    for attempt in range(retries + 1):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt
            )

            if response.text and response.text.strip():
                return response.text.strip()

            return "The AI returned an empty response. Please try again."

        except Exception as error:
            if attempt < retries:
                time.sleep(2 * (attempt + 1))
            else:
                # Avoid printing the API key or full request details
                return (
                    "Sorry, the AI request failed. Check your internet, "
                    "API key, model access and usage limits. "
                    "Technical error type: " + type(error).__name__
                )

# Sample evaluation cases
evaluation_cases = [
    {
        "question": "What is photosynthesis? Include the words light and plants.",
        "expected": ["photosynthesis", "light"]
    },
    {
        "question": "What is a variable in Python? Include the word store.",
        "expected": ["variable", "store"]
    },
    {
        "question": "Explain gravity in simple words and mention mass.",
        "expected": ["gravity", "mass"]
    }
]

print("\nRunning evaluation examples...")
print("-" * 45)

passed = 0

for number, case in enumerate(evaluation_cases, start=1):
    answer = generate_answer(case["question"])
    answer_lower = answer.lower()

    # Basic keyword check, not a full measure of answer quality
    keyword_matches = sum(
        word in answer_lower for word in case["expected"]
    )
    test_passed = keyword_matches == len(case["expected"])

    if test_passed:
        passed += 1

    print(f"\nTest {number}: {'PASS' if test_passed else 'CHECK'}")
    print("Question:", case["question"])
    print("Answer:", answer)
    print("Keyword matches:",
          keyword_matches, "/", len(case["expected"]))

print("\nEvaluation summary:", passed, "/", len(evaluation_cases),
      "tests passed the keyword check.")
print("Note: A keyword match does not guarantee factual correctness.")

# Interactive question-and-answer demo
print("\n" + "=" * 45)
print("AI STUDY ASSISTANT IS READY!")
print("Type 'exit' or 'quit' to stop.")
print("=" * 45)

while True:
    try:
        question = input("\nYour study question: ").strip()

        if question.lower() in ["exit", "quit"]:
            print("Session ended. Thank you!")
            break

        answer = generate_answer(question)
        print("\nAI Answer:\n", answer)

    except (KeyboardInterrupt, EOFError):
        print("\nSession ended.")
        break

AI Study Assistant - SWYNEX Task 3
Paste your Gemini API key (hidden): ··········

Checking Gemini model access...
Connected successfully.
Selected model: gemini-3-flash-preview

Running evaluation examples...
---------------------------------------------

Test 1: PASS
Question: What is photosynthesis? Include the words light and plants.
Answer: Hello! I'm your AI Study Assistant, and I'd be happy to help you understand this amazing process.

**Photosynthesis** is how **plants** make their own food. Unlike humans, who have to eat food to get energy, plants use **light** to "cook" their own meals!

Here is a step-by-step look at how it works:

1.  **Catching the Light:** Plants have a special green substance in their leaves called *chlorophyll*. This acts like a solar panel to catch energy from sunlight.
2.  **Gathering Ingredients:** To make food, the plant needs two more things: water (from the soil) and carbon dioxide (a gas from the air).
3.  **The Mixing Process:** Inside the leaf,